In [ ]:
# 실험 제목
- 담당: 이채원 
- 날짜: 2026-09-22
- 목적: 데이터 성분 추출 연습해보기 

> 끝나면 결과를 `experiments/LOG.md`에 한 줄 남기기

In [ ]:
import json
from pathlib import Path

import pandas as pd

# VS Code Jupyter는 노트북이 있는 폴더를 cwd로 잡으므로, pyproject.toml이 있는
# 레포 최상위 폴더를 직접 찾아 올라간다.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW_DIR = REPO_ROOT / "data/raw/Training/02.라벨링데이터"
PROCESSED_PATH = REPO_ROOT / "data/processed/qa_flat.jsonl"

if not RAW_DIR.exists():
    raise FileNotFoundError(f"{RAW_DIR} 를 찾을 수 없습니다.")

raw_files = sorted(RAW_DIR.glob("**/*.json"))
print(f"원본 라벨링 파일 수: {len(raw_files)}")

records = []
for path in raw_files:
    with open(path, encoding="utf-8") as f:
        item = json.load(f)

    source = item.get("source", {})
    consulting = item.get("consulting", {})

    for qa in item.get("qa_data", []):
        input_ = qa.get("input", {})
        records.append({
            "qa_id": qa.get("qa_id"),
            "source_institution": source.get("source_institution"),
            "consulting_category": consulting.get("consulting_category"),
            "consulting_topic": consulting.get("consulting_topic"),
            "task_category": qa.get("task_category"),
            "consulting_situation": qa.get("consulting_situation"),
            "qa_topic": qa.get("qa_topic"),
            "consulting_purpose": qa.get("consulting_purpose"),
            "core_financial_terms": qa.get("core_financial_terms"),
            "instruction": qa.get("instruction"),
            "question": input_.get("question"),
            "answer": input_.get("answer"),
            "follow_up_question": input_.get("follow_up_question"),
            "output": qa.get("output"),
        })

df = pd.DataFrame(records)

# data/processed/qa_flat.jsonl 이 아직 없으므로 여기서 새로 만들어서 저장
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_json(PROCESSED_PATH, orient="records", lines=True, force_ascii=False)

df.head()

## 관찰 / 메모
